In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
#from langchain_community.chat_models import ChatOllama
from langchain_ollama import ChatOllama

# --- LLM ---
llm = ChatOllama(model="llama3")



In [2]:
# --- State ---
class ChatState(TypedDict):
    message: str
    response: str
    intent: str

# --- Nodes ---

def intent_detection(state: ChatState):
    text = state["message"].lower()

    if "match" in text:
        intent = "matchmaking"
    else:
        intent = "general"

    return {**state, "intent": intent}


def general_chat(state: ChatState):
    prompt = f"User: {state['message']}\nAssistant:"
    res = llm.invoke(prompt)

    return {**state, "response": res.content}


def matchmaking_node(state: ChatState):
    prompt = f"""
    You are a matrimonial assistant.
    Analyze this request and respond helpfully:

    {state['message']}
    """
    res = llm.invoke(prompt)

    return {**state, "response": res.content}


# --- Router ---
def route(state: ChatState):
    return state["intent"]


# --- Graph ---
builder = StateGraph(ChatState)

builder.add_node("intent_detection", intent_detection)
builder.add_node("general", general_chat)
builder.add_node("matchmaking", matchmaking_node)

builder.set_entry_point("intent_detection")

builder.add_conditional_edges(
    "intent_detection",
    route,
    {
        "general": "general",
        "matchmaking": "matchmaking",
    },
)

builder.add_edge("general", END)
builder.add_edge("matchmaking", END)

graph = builder.compile()



In [3]:

user_input = "Match these two profiles based on values and career goals"

result = graph.invoke({"message": user_input})

print(result["response"])

Exciting! As a matrimonial assistant, my goal is to help you find your perfect match. I'd be happy to analyze the two profiles and provide some insights.

To get started, could you please share the two profiles with me? These can include information such as:

* Career goals (e.g., industry, job title, aspirations)
* Values (e.g., family-oriented, adventurous, spiritual)
* Personality traits (e.g., optimistic, analytical, creative)
* Education background
* Interests and hobbies

The more information you provide, the better I can match these two profiles based on their values and career goals.

Once I have this information, I'll use my expertise to analyze the compatibility of these two individuals. This may involve identifying commonalities in:

* Shared values and priorities
* Complementary strengths and weaknesses
* Career paths that align with each other's goals

I'll then provide a detailed report highlighting the potential match between these two profiles. This will help you unders